# Packages import

In [4]:
import requests
import os
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

0. Prepare selenium


In [ ]:
product_code = "174130671"
page = 1
url = f"https://www.ceneo.pl/{product_code}/opinie-{page}"
path_to_driver = "D:\\chromedriver-win64\\chromedriver-win64\\chromedriver.exe"
s = Service(path_to_driver)
driver = webdriver.Chrome(service=s)
driver.get(url)
driver.maximize_window()
driver.find_element(by="xpath", value="//*[@id="js_cookie-consent-general"]/div/div[2]/button[1]").click()

1. Provide url of the products opinions page

In [1]:
##product_code = "174130671"
##page = 1
##url = f"https://www.ceneo.pl/{product_code}/opinie-{page}"

2. Send request to provided url

In [40]:
response = requests.get(url)
print(response.status_code)

200


3. Fetch product name

In [41]:
page_dom = BeautifulSoup(response.text, "html.parser")
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [42]:
product_name = page_dom.find("h1").get_text()
print(product_name)
print(type(product_name))

Doppelherz Minoxidil Dla Mężczyzn 60g
<class 'str'>


4. Fetch all opinions from the webpage

In [43]:
opinions = page_dom.select('div.js_product-review:not(.user-post--highlight)')
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
10


5. Parse opinions to extract required data

In [44]:
all_opinions = []
for opinion in opinions:
    single_opinion = {
        'opinion_id': opinion.get("data-entry-id"),
        'author': opinion.select_one("span.user-post__author-name").get_text().strip(),
        'recommendation': opinion.select_one('span.user-post__author-recomendation > em').get_text() if opinion.select_one('span.user-post__author-recomendation > em') else None,
        'score': opinion.select_one('span.user-post__score-count').get_text(),
        'content': opinion.select_one('div.user-post__text').get_text(),
        'pros': [p.get_text() for p in opinion.select('div.review-feature__item--positive')],
        'cons': [c.get_text() for c in opinion.select('div.review-feature__item--negative')],
        'helpful': opinion.select_one('button.vote-yes > span').get_text(),
        'unhelpful': opinion.select_one('button.vote-no > span').get_text(),
        'publish_date': opinion.select_one('span.user-post__published > time:nth-child(1)[datetime]').get('datetime'),
        'purchase_date': opinion.select_one('span.user-post__published > time:nth-child(2)[datetime]').get('datetime') if opinion.select_one('span.user-post__published > time:nth-child(2)[datetime]') else None
    }
    all_opinions.append(single_opinion)

In [45]:
print(all_opinions)

[{'opinion_id': '19959382', 'author': 'l...o', 'recommendation': 'Polecam', 'score': '5/5', 'content': 'Na chwilę obecną mogę napisać jedynie tyle... Produkt łatwy w aplikacji, pozostawia włosy nieco sztywne, ale to raczej normalne. Na efekty, które mam nadzieję się pojawią trzeba poczekać. ', 'pros': [], 'cons': [], 'helpful': '0', 'unhelpful': '2', 'publish_date': '2025-09-24 20:22:56', 'purchase_date': '2025-09-21 19:59:06'}, {'opinion_id': '20403013', 'author': 'p...k', 'recommendation': 'Polecam', 'score': '5/5', 'content': 'Strzygę się teraz trzy razy dziennie. Zwykłe nożyczki nie dają rady. Muszę używać podkaszarki spalinowej. W cichym pomieszczeniu słychać jak rosną.', 'pros': [], 'cons': [], 'helpful': '1', 'unhelpful': '2', 'publish_date': '2026-03-23 12:57:46', 'purchase_date': '2026-03-11 21:45:32'}, {'opinion_id': '19689180', 'author': 's...0', 'recommendation': 'Polecam', 'score': '5/5', 'content': 'Efekt jest, ale potrzeba cierpliwości.', 'pros': [], 'cons': [], 'helpful

6. Check if there is next page with opinions

In [46]:
driver.find_element(by="xpath", value="//*[@id="js_cookie-consent-general"]/div/div[2]/button[1]").click()

8. Save acquired opinions

In [47]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [48]:
with open(f'./opinions/{product_code}.json','w', encoding="UTF-8") as jf:
    json.dump(all_opinions, jf, indent=4, ensure_ascii=False)